In [ ]:
import os
from pathlib import Path

# Load HF_TOKEN from .env (lines look like `export HF_TOKEN=...`)
for line in Path('.env').read_text().splitlines():
    line = line.strip().removeprefix('export ')
    if '=' in line and not line.startswith('#'):
        key, value = line.split('=', 1)
        os.environ[key.strip()] = value.strip().strip('"\'')

from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceFW/fineweb",
    "CC-MAIN-2025-08",
    split="train",
    streaming=True,
)

In [ ]:
NUM_DOCS = 1000  # the stream is huge, so only take a sample

vespa_feed = []

for doc in dataset.take(NUM_DOCS):
    # FineWeb ids look like '<urn:uuid:d66bc6fe-...>'; strip the wrapper so it is a clean Vespa doc id
    doc_id = doc['id'].removeprefix('<urn:uuid:').removesuffix('>')
    vespa_feed.append({
        'id': doc_id,
        'fields': {
            'id': doc_id,
            'url': doc['url'],
            'text': doc['text'],
            'dump': doc['dump'],
            'year': int(doc['date'][:4]),  # date is e.g. '2013-05-18T05:48:54Z'
            'token_count': doc['token_count'],
        }
    })

print(len(vespa_feed))
print(vespa_feed[0]['fields']['url'])

In [ ]:
from vespa.application import ApplicationPackage

app_package = ApplicationPackage(name='fineweb')

In [ ]:
from vespa.package import Schema, Document

film_schema = Schema(name='fineweb_schema', document=Document())

In [ ]:
from vespa.package import Field

film_schema.add_fields(
    Field(
        name='url',
        type='string',
        indexing=['index', 'summary'],
        index="enable-bm25",
    ),
    Field(
        name='text',
        type='string',
        indexing=['index', 'summary'],
        index="enable-bm25",
    ),
    Field(
        name='dump',
        type='string',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='year',
        type='int',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='token_count',
        type='int',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='id',
        type='string',
        indexing=['summary']
    )
)

In [ ]:
from vespa.package import FieldSet, RankProfile

film_schema.add_field_set(
    FieldSet(
        name='default',
        fields=['url', 'text']
    ))

# Rank on both fields, boosting url matches a bit
film_schema.add_rank_profile(
    RankProfile(
        name='default',
        first_phase='bm25(text) + 2 * bm25(url)'
    ))

In [ ]:
app_package.add_schema(film_schema)

In [ ]:
from vespa.deployment import VespaCloud

vespa_cloud = VespaCloud(
    tenant='akashsinghal',
    application='fineweb',
    application_package=app_package
)

In [ ]:
app = vespa_cloud.deploy(region="aws-euw1-az1")

In [ ]:
cert_path = app.cert
key_path = app.key

endpoint = vespa_cloud.get_mtls_endpoint(region="aws-euw1-az1")

In [ ]:
from vespa.application import Vespa

vespa_application = Vespa(endpoint, cert=cert_path, key=key_path)

In [ ]:
from vespa.io import VespaResponse

def callback(response: VespaResponse, id:str):
    if not response.is_successful():
        print(f"Error when feeding document {id}: {response.get_json()}")

vespa_application.feed_iterable(vespa_feed, schema='fineweb_schema', callback=callback)

In [ ]:
from vespa.io import VespaQueryResponse

def print_hits(response: VespaQueryResponse):
    for i, hit in enumerate(response.hits):
        f = hit['fields']
        print(f'{i+1:>3}  {f["id"][:8]}  {f["year"]}  {f["token_count"]:>6} tok  {f["url"]}')

In [ ]:
# Searching for 'tornado' in the default fieldset (url + text)

response = vespa_application.query(
    yql="select * from sources * where default contains 'tornado'", # The default limit in Vespa is 10 results
)

print_hits(response)

In [ ]:
# Using userQuery(): the query text is passed as a separate parameter

response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 100",
    query="tornado damage",
)

print_hits(response)

In [ ]:
# Search by text: matches words in the url or the body
response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 10",
    query="tornado damage",
)

print_hits(response)

In [ ]:
# Search by url: pass a full url or a fragment; punctuation is tokenized away
response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 10",
    query="http://ap.org/Content/Press-Release/2012/How-AP-reported-in-all-formats-from-tornado-stricken-regions",
    type="any",  # match any term, ranked by how many match
)

print_hits(response)

In [ ]:
# Restrict to one field when you only want url or only text
response = vespa_application.query(
    yql="select * from sources * where url contains 'ap' and text contains 'tornado' limit 10",
)

print_hits(response)